# SI4006 · Sesión 11 · Lab: **El transformer aprende a ver — CLIP y un LLM multimodal**

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 4

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manularrea/EAFIT-SI4006/blob/main/sesiones/s11/S11_Lab_Vision.ipynb)

---

Hoy el mismo transformer entra a las imágenes. Dos labs:

- **Lab A · CLIP zero-shot:** texto e imagen en el mismo espacio. Clasificar sin entrenar, y buscar imágenes con una frase (Vision RAG en 5 líneas).
- **Lab B · Visual QA:** un LLM que mira (Qwen2-VL-2B, la receta de LLaVA en tamaño Colab). Describir, preguntar con válvula de escape, extraer JSON — y una pregunta trampa.

> **Traigan 3–6 imágenes de SU dominio** (fotos de producto, hojas de cultivo, documentos escaneados, mascotas…). El notebook trae imágenes de ejemplo para que corra solo; donde diga, pongan las suyas.
>
> Corre en Colab con GPU **T4** (Entorno de ejecución → Cambiar tipo → T4). En CPU el Lab B es lento.

## 0 · Setup

In [ ]:
# Colab ya trae transformers y torch. Instalamos lo que falta.
%pip install -q "transformers>=4.45" accelerate pillow qwen-vl-utils
print('Listo.')

In [ ]:
import torch, requests, io, json, re
from PIL import Image
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu': print('⚠️  Sin GPU: el Lab A va bien; el Lab B (modelo multimodal) será lento.')

## 1 · Las imágenes

Tres de ejemplo (dominio público, vía COCO) para que el notebook corra. **Reemplácenlas por las suyas**: suban archivos al panel de la izquierda y cambien las rutas, o peguen URLs.

In [ ]:
def cargar(src):
    if src.startswith('http'):
        return Image.open(io.BytesIO(requests.get(src, timeout=30).content)).convert('RGB')
    return Image.open(src).convert('RGB')

# 👉 SUS IMÁGENES AQUÍ: {'nombre corto': 'URL o ruta'}
IMAGENES = {
    'gatos':   'http://images.cocodataset.org/val2017/000000039769.jpg',
    'sala':    'http://images.cocodataset.org/val2017/000000000139.jpg',
    'esqui':   'http://images.cocodataset.org/val2017/000000000785.jpg',
}
imgs = {k: cargar(v) for k, v in IMAGENES.items()}
for k, im in imgs.items(): print(f'{k:8s}', im.size)
# vista rápida
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 3))
for ax, (k, im) in zip(axs, imgs.items()): ax.imshow(im); ax.set_title(k); ax.axis('off')
plt.show()

## 2 · Lab A · CLIP: texto e imagen en el mismo espacio

CLIP tiene dos encoders (imagen → vector, texto → vector) entrenados para que la foto y su descripción queden cerca. Vamos a:

1. vectorizar las imágenes y unos **prompts-clase** escritos por nosotros,
2. calcular el **coseno** imagen–texto (el mismo de S05, S06 y S07),
3. ver la matriz completa y quién gana por fila (zero-shot),
4. cambiar la formulación del prompt y ver si mueve la aguja,
5. buscar una imagen con una frase (retrieval sobre fotos).

In [ ]:
from transformers import CLIPModel, CLIPProcessor
CLIP_ID = 'openai/clip-vit-base-patch32'   # ViT-B/32: el encoder de imagen es un ViT con parches de 32 px
clip = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
proc = CLIPProcessor.from_pretrained(CLIP_ID)
print('CLIP listo:', CLIP_ID)

In [ ]:
@torch.no_grad()
def emb_imagenes(lista_pil):
    ent = proc(images=lista_pil, return_tensors='pt').to(device)
    v = clip.get_image_features(**ent)
    return torch.nn.functional.normalize(v, dim=-1)      # normalizar → el producto punto ES el coseno

@torch.no_grad()
def emb_textos(lista_str):
    ent = proc(text=lista_str, return_tensors='pt', padding=True, truncation=True).to(device)
    v = clip.get_text_features(**ent)
    return torch.nn.functional.normalize(v, dim=-1)

# 👉 SUS CLASES AQUÍ (en inglés rinde mejor). Una por línea, como frase natural.
CLASES = ['a photo of cats on a couch', 'a photo of a living room', 'a photo of a person skiing', 'a photo of a dog']

V_img = emb_imagenes(list(imgs.values()))
V_txt = emb_textos(CLASES)
print('vectores imagen:', tuple(V_img.shape), '| vectores texto:', tuple(V_txt.shape))

In [ ]:
# TODO A1: matriz de cosenos imagen × texto. Pista: los vectores están normalizados.
def matriz_cosenos(V_img, V_txt):
    return None   # ← reemplacen

S = matriz_cosenos(V_img, V_txt)

def mostrar_matriz(S, filas, cols):
    print(f'{"":10s}' + ''.join(f'{c[:18]:>20s}' for c in cols))
    for i, f in enumerate(filas):
        print(f'{f:10s}' + ''.join(f'{S[i, j].item():20.3f}' for j in range(len(cols))))
        print(f'{"":10s}→ gana: {cols[S[i].argmax().item()]}')
mostrar_matriz(S, list(imgs.keys()), CLASES)

> 👀 **Miren la diagonal.** Nadie entrenó nada con estas fotos: el contraste sobre 400 M pares ya dejó cerca cada imagen de su descripción. ¿Y la clase que no está en ninguna foto ("a photo of a dog")? Debería perder en todas las filas — verifíquenlo.

### A2 · El prompt es un hiperparámetro

Misma clase, tres formulaciones. ¿Cambia el coseno? ¿Cambia el ganador?

In [ ]:
# TODO A2: para UNA de sus imágenes, comparen tres formulaciones del prompt de su clase correcta.
imagen = 'gatos'                             # ← una de sus claves
formulaciones = ['cats', 'a photo of cats', 'a close-up photo of two cats sleeping on a sofa']   # ← edítenlas

v = emb_imagenes([imgs[imagen]])
for f in formulaciones:
    cos = None   # ← coseno entre v y el embedding de f
    print(f'{cos:.3f}  {f}')

### A3 · Buscar imágenes con una frase (Vision RAG en 5 líneas)

Es el retrieval de S07: la consulta se vuelve vector, y se devuelven los vectores más cercanos. Solo que ahora los "chunks" son fotos.

In [ ]:
def buscar_imagen(consulta, k=2):
    q = emb_textos([consulta])
    scores = (V_img @ q.T).squeeze(1)
    top = scores.argsort(descending=True)[:k]
    return [(list(imgs.keys())[i], round(scores[i].item(), 3)) for i in top]

for consulta in ['animals resting', 'snow and mountains', 'furniture']:   # ← consultas de su dominio
    print(f'{consulta:22s} →', buscar_imagen(consulta))

> 👀 **De vuelta del Lab A:** anoten (1) la clase donde CLIP se equivocó o dudó, (2) qué formulación de prompt ganó. Lo primero es su primer caso de prueba visual para M4; lo segundo es la evidencia de que el prompt se mide, no se supone.

## 3 · Lab B · Un LLM que mira: visual QA

La receta de LLaVA: encoder visual (ViT de CLIP) → proyector → tokens visuales que se concatenan con el texto → LLM decoder. Usamos **Qwen2-VL-2B-Instruct**, que cabe en la T4. (LLaVA-1.5-7B en 4 bits queda como celda opcional al final.)

Tres tipos de prompt, y luego una trampa.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
VLM_ID = 'Qwen/Qwen2-VL-2B-Instruct'
vlm = Qwen2VLForConditionalGeneration.from_pretrained(VLM_ID, torch_dtype=torch.float16 if device=='cuda' else torch.float32).to(device).eval()
vproc = AutoProcessor.from_pretrained(VLM_ID)
print('VLM listo:', VLM_ID)

@torch.no_grad()
def preguntar(imagen_pil, prompt, system=None, max_new_tokens=160):
    msgs = []
    if system: msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': [{'type': 'image', 'image': imagen_pil}, {'type': 'text', 'text': prompt}]})
    texto = vproc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    im_in, vid_in = process_vision_info(msgs)
    ent = vproc(text=[texto], images=im_in, videos=vid_in, padding=True, return_tensors='pt').to(device)
    out = vlm.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False)
    return vproc.batch_decode(out[:, ent['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()

In [ ]:
imagen = imgs['gatos']     # ← una de las suyas

# B1 · describir
print('DESCRIBIR →', preguntar(imagen, 'Describe la imagen en dos frases, en español.'))

In [ ]:
# B2 · preguntar CON válvula de escape (la misma idea del prompt aumentado de S07)
SYSTEM_ESCAPE = ('Responde SOLO con lo que se ve en la imagen. '
                 'Si algo no se aprecia en la imagen, responde exactamente: "No se aprecia en la imagen."')
print('CON VÁLVULA →', preguntar(imagen, '¿Cuántos animales hay y de qué color son?', system=SYSTEM_ESCAPE))

In [ ]:
# B3 · salida estructurada: lo que sale del VLM entra a su sistema como texto
prompt_json = ('Devuelve SOLO un JSON con los campos: '
               '{"objetos": [lista de objetos principales], "escena": "interior|exterior", "texto_visible": "texto legible o null"}')
salida = preguntar(imagen, prompt_json, system=SYSTEM_ESCAPE)
print(salida)
# TODO B3: parseen el JSON con cuidado (como en el lab A de S10). Si falla, devuelvan None y díganlo.
def parsear_json(t):
    return None   # ← reemplacen
print('parseado:', parsear_json(salida))

### B4 · La pregunta trampa

Pregunten por algo que **NO** está en la imagen (un número de placa, una fecha, el nombre de una persona, la raza exacta). Comparen la respuesta **sin** y **con** válvula de escape.

In [ ]:
# TODO B4: escriban su pregunta trampa para SU imagen.
trampa = '¿Qué fecha aparece escrita en la esquina inferior derecha de la imagen?'   # ← edítenla

print('SIN válvula →', preguntar(imagen, trampa))
print('CON válvula →', preguntar(imagen, trampa, system=SYSTEM_ESCAPE))

> 👀 **De vuelta del Lab B:** ¿alucinó sin válvula? ¿La válvula lo frenó? Un VLM alucina como un LLM porque ES un LLM. Esa pregunta trampa es un caso adversarial para su eval set de M4 — igual que los de S06.

### Opcional · LLaVA-1.5-7B en 4 bits (solo con GPU)

La misma receta, el modelo del paper original. Cabe en la T4 cuantizado (S15 explica qué es eso).

In [ ]:
# %pip install -q bitsandbytes
# from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
# llava = LlavaForConditionalGeneration.from_pretrained('llava-hf/llava-1.5-7b-hf', quantization_config=bnb, device_map='auto')
# lproc = AutoProcessor.from_pretrained('llava-hf/llava-1.5-7b-hf')
# conv = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'Describe the image in two sentences.'}]}]
# prompt = lproc.apply_chat_template(conv, add_generation_prompt=True)
# ent = lproc(images=imagen, text=prompt, return_tensors='pt').to(llava.device, torch.float16)
# out = llava.generate(**ent, max_new_tokens=120, do_sample=False)
# print(lproc.decode(out[0][ent['input_ids'].shape[1]:], skip_special_tokens=True))

## Lo que se llevan

- **CLIP** mide parecido entre una foto y una frase con el coseno de siempre. Clasificar = escribir las clases como texto. Buscar = el retrieval de S07 sobre fotos.
- **Un VLM** es un LLM cuyo prompt incluye tokens que vienen de una imagen: describe, responde, extrae JSON… y alucina igual. Válvula de escape, salida validada y casos trampa: todo aplica.
- Para **M4** (S13 · cierra dom 25 OCT): un componente visual, un patrón (input / tool / output) y ≥5 casos con imagen en el eval set. Lo que anotaron hoy (dónde falló CLIP, la pregunta trampa) son los primeros.

*SI4006 · Universidad EAFIT · Sesión 11.*